# HS524M Test 2 — Question 1: QField waste-bin observations

This notebook reads a QField GeoPackage, creates an interactive Folium point map with observation details, collector attribution and photographs, and saves a portable folder for a personal webpage. **Run cells from top to bottom.** The source GeoPackage and photos must already exist on your machine. Edit the four configuration values in Section 1 before running.

The deliverable is `waste_bin_webmap/waste_bin_map.html` together with its `photos` subfolder. Upload the **whole folder** to the website, preserving the `photos/` path. Open the hosted map and click several markers to verify the photographs load. Leaflet and OpenStreetMap also require an internet connection in the viewer's browser.

In [1]:
# Install once if required. Restart the kernel if Jupyter asks.
%pip install -q geopandas pyogrio folium pandas pillow

Note: you may need to restart the kernel to use updated packages.


## 1. Paths and attribution

Enter your actual full name at the prompt, or set `MY_NAME` in the configuration cell. `DEFAULT_COLLECTOR` may be set to that name **only if every observation without a collector field was collected by you**. If classmates' records are included, add a collector column with their original names or provide a verified ID-to-name mapping below. Unattributed records stop the export.

In [2]:
from pathlib import Path
from urllib.parse import unquote, quote
import hashlib
import html
import json
import re
import shutil

import folium
import geopandas as gpd
import pandas as pd
import pyogrio
from IPython.display import display

# Change these paths for your computer. These are the paths used by the uploaded draft.
GPKG_PATH = Path(r"C:\Users\jrana\Downloads\notes.gpkg")
PHOTO_DIR = Path(r"C:\Users\jrana\Desktop\PHOTOS")
OUTPUT_DIR = Path(r"C:\Users\jrana\Downloads\waste_bin_webmap")
MY_NAME = ""  # optional: enter your full name here, or supply it at the prompt
DEFAULT_COLLECTOR = ""  # only if every otherwise-unattributed record is yours

# Optional explicit overrides. Leave None to detect columns by likely names.
LAYER_NAME = None
TITLE_COL = None
DESCRIPTION_COL = None
PHOTO_COL = None
COLLECTOR_COL = None
ID_COL = None

# For contributor identity encoded as IDs instead of names, map verified values here.
COLLECTOR_NAME_BY_ID = {}  # e.g. {"u123": "Actual Student Name"}
REQUIRE_PHOTO_FOR_EVERY_RECORD = True

if not GPKG_PATH.is_file():
    raise FileNotFoundError(f"GeoPackage not found: {GPKG_PATH}. Edit GPKG_PATH above.")
if not PHOTO_DIR.is_dir():
    raise FileNotFoundError(f"Photo directory not found: {PHOTO_DIR}. Edit PHOTO_DIR above.")
if not MY_NAME.strip() or MY_NAME.strip().lower() in {"your name", "name"}:
    MY_NAME = input("Enter your full name for map attribution: ").strip()
if not MY_NAME or MY_NAME.lower() in {"your name", "name"}:
    raise ValueError("A real full name is required to credit your observations.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEB_PHOTOS = OUTPUT_DIR / "photos"
WEB_PHOTOS.mkdir(parents=True, exist_ok=True)
print("GeoPackage:", GPKG_PATH)
print("Photographs:", PHOTO_DIR)
print("Output:", OUTPUT_DIR)

Enter your full name for map attribution:  Vidya Anand


GeoPackage: C:\Users\jrana\Downloads\notes.gpkg
Photographs: C:\Users\jrana\Desktop\PHOTOS
Output: C:\Users\jrana\Downloads\waste_bin_webmap


## 2. Inspect and load the QField point layer

The notebook selects the first nonempty Point/MultiPoint layer if no layer name is supplied. Check the displayed attributes. Override the column settings above if the inferred title, description, photo or collector field is wrong. All coordinates are transformed to WGS84 for Folium.

In [3]:
layers = pyogrio.list_layers(str(GPKG_PATH))
print("GeoPackage layers (name, geometry):")
for name, geom in layers:
    print("  ", name, geom)

if LAYER_NAME is None:
    point_layers = [name for name, geom in layers if geom and "Point" in str(geom)]
    if not point_layers:
        raise ValueError("No point layer found. Inspect QGIS/QField GeoPackage layers.")
    LAYER_NAME = point_layers[0]
if LAYER_NAME not in [name for name, _ in layers]:
    raise ValueError(f"Layer {LAYER_NAME!r} not present in the GeoPackage.")

raw = gpd.read_file(GPKG_PATH, layer=LAYER_NAME, engine="pyogrio")
if raw.crs is None:
    raise ValueError("Layer CRS is missing. Set the correct source CRS in QGIS before running.")
raw = raw.loc[raw.geometry.notna() & ~raw.geometry.is_empty].copy()
raw = raw[raw.geometry.geom_type.isin(["Point", "MultiPoint"])].copy()
if raw.empty:
    raise ValueError("The selected layer has no nonempty point observations.")
points = raw.to_crs(epsg=4326).explode(index_parts=False).reset_index(drop=True)
points = points[points.geometry.geom_type.eq("Point")].copy().reset_index(drop=True)
if points.empty:
    raise ValueError("No valid point observations remain after geometry conversion.")
print(f"Selected {LAYER_NAME!r}: {len(raw)} records, {len(points)} points; source CRS {raw.crs}")
print("Available columns:", [c for c in points.columns if c != points.geometry.name])
display(points.drop(columns=points.geometry.name).head(5))

GeoPackage layers (name, geometry):
   notes_point Point Z
   notes_line LineString Z
   notes_polygon Polygon Z
   notes_attachments None
Selected 'notes_point': 43 records, 43 points; source CRS EPSG:4326
Available columns: ['uuid', 'color', 'title', 'note', 'timestamp']


,uuid,color,title,note,timestamp
0,{be965bb7-aeb2-45c6-a5c1-6d91de88a90e},#377eb8,Bin_1,Segregated,2026-09-01 16:39:19.769000+00:00
1,{ebe25330-4aab-4dd0-b78a-63b47fc9d8fb},#377eb8,Bin_2,Segregated,2026-09-01 16:49:39.914000+00:00
2,{7234b314-4271-456a-833d-890c5bde1b09},#377eb8,Bin_3,Non - Segregated,2026-09-01 16:54:26.400000+00:00
3,{92b5c328-1fe9-42d6-8719-d4b63aed3281},#377eb8,Bin_4,Non - Segregated,2026-09-01 16:57:56.175000+00:00
4,{04bb926e-1998-4f28-859e-5c303b1fb1b3},#377eb8,NaN,NaN,2026-09-02 17:14:48.054000+00:00


In [5]:

# Inspect all columns in the QField point layer
print("All available columns:")
for col in points.columns:
    print(repr(col))

print("\nFirst 5 records:")
display(points.head())

print("\nPossible photo or attachment columns:")
photo_candidates = [
    col for col in points.columns
    if any(word in str(col).lower()
           for word in ["photo", "image", "picture", "attach", "media", "file"])
]
print(photo_candidates if photo_candidates else "No matching photo column found.")

print("\nPhotograph directory:", PHOTO_DIR)
print("Files in photograph directory:")
for file in PHOTO_DIR.rglob("*"):
    if file.is_file():
        print(file.name)


All available columns:
'uuid'
'color'
'title'
'note'
'timestamp'
'geometry'

First 5 records:


,uuid,color,title,note,timestamp,geometry
0,{be965bb7-aeb2-45c6-a5c1-6d91de88a90e},#377eb8,Bin_1,Segregated,2026-09-01 16:39:19.769000+00:00,POINT Z (79.59487 13.70819 0)
1,{ebe25330-4aab-4dd0-b78a-63b47fc9d8fb},#377eb8,Bin_2,Segregated,2026-09-01 16:49:39.914000+00:00,POINT Z (79.59456 13.70713 0)
2,{7234b314-4271-456a-833d-890c5bde1b09},#377eb8,Bin_3,Non - Segregated,2026-09-01 16:54:26.400000+00:00,POINT Z (79.59489 13.70659 0)
3,{92b5c328-1fe9-42d6-8719-d4b63aed3281},#377eb8,Bin_4,Non - Segregated,2026-09-01 16:57:56.175000+00:00,POINT Z (79.59446 13.70604 0)
4,{04bb926e-1998-4f28-859e-5c303b1fb1b3},#377eb8,NaN,NaN,2026-09-02 17:14:48.054000+00:00,POINT Z (79.59422 13.70691 0)



Possible photo or attachment columns:
No matching photo column found.

Photograph directory: C:\Users\jrana\Desktop\PHOTOS
Files in photograph directory:


In [8]:
PHOTO_COL = "photo_url"

## 3. Resolve photographs and collector names

Photos may be referenced by relative QField paths, absolute local paths or filenames. The resolver checks the listed roots, then looks up unique filenames recursively. Duplicate filenames are treated as ambiguous unless the recorded path identifies one. Images are copied with unique output names so two observations cannot overwrite one another.

In [11]:

import sqlite3
from pathlib import Path

# Inspect tables stored inside the GeoPackage
con = sqlite3.connect(str(GPKG_PATH))

tables = pd.read_sql_query(
    "SELECT name, type FROM sqlite_master "
    "WHERE type IN ('table', 'view') ORDER BY name",
    con
)

print("Tables and views inside GeoPackage:")
display(tables)

# Check metadata for photo or attachment fields
for table_name in tables["name"]:
    if table_name.startswith("sqlite_"):
        continue

    try:
        columns = pd.read_sql_query(
            f'PRAGMA table_info("{table_name}")', con
        )
        print(f"\n{table_name}: {columns['name'].tolist()}")
    except Exception as e:
        print(f"Could not inspect {table_name}: {e}")

con.close()

# Display sample photo filenames
print("\nFirst 30 image filenames:")
for p in sorted(photos, key=lambda p: p.name)[:30]:
    print(p.name)

# Show bin titles and UUIDs
print("\nWaste-bin records:")
display(points[["uuid", "title", "note"]].head(15))



Tables and views inside GeoPackage:


,name,type
0,gpkg_contents,table
1,gpkg_extensions,table
2,gpkg_geometry_columns,table
3,gpkg_ogr_contents,table
4,gpkg_spatial_ref_sys,table
5,gpkg_tile_matrix,table
6,gpkg_tile_matrix_set,table
7,notes_attachments,table
8,notes_line,table
9,notes_point,table



gpkg_contents: ['table_name', 'data_type', 'identifier', 'description', 'last_change', 'min_x', 'min_y', 'max_x', 'max_y', 'srs_id']

gpkg_extensions: ['table_name', 'column_name', 'extension_name', 'definition', 'scope']

gpkg_geometry_columns: ['table_name', 'column_name', 'geometry_type_name', 'srs_id', 'z', 'm']

gpkg_ogr_contents: ['table_name', 'feature_count']

gpkg_spatial_ref_sys: ['srs_name', 'srs_id', 'organization', 'organization_coordsys_id', 'definition', 'description']

gpkg_tile_matrix: ['table_name', 'zoom_level', 'matrix_width', 'matrix_height', 'tile_width', 'tile_height', 'pixel_x_size', 'pixel_y_size']

gpkg_tile_matrix_set: ['table_name', 'srs_id', 'min_x', 'min_y', 'max_x', 'max_y']

notes_attachments: ['fid', 'note_layer', 'note_uuid', 'media', 'description', 'timestamp']

notes_line: ['fid', 'geom', 'uuid', 'color', 'title', 'note', 'timestamp']

notes_point: ['fid', 'geom', 'uuid', 'color', 'title', 'note', 'timestamp']

notes_polygon: ['fid', 'geom', 'uuid',

,uuid,title,note
0,{be965bb7-aeb2-45c6-a5c1-6d91de88a90e},Bin_1,Segregated
1,{ebe25330-4aab-4dd0-b78a-63b47fc9d8fb},Bin_2,Segregated
2,{7234b314-4271-456a-833d-890c5bde1b09},Bin_3,Non - Segregated
3,{92b5c328-1fe9-42d6-8719-d4b63aed3281},Bin_4,Non - Segregated
4,{04bb926e-1998-4f28-859e-5c303b1fb1b3},NaN,NaN
5,{16dfbf27-c878-40e1-bb5a-bfc9358b2eb0},NaN,NaN
6,{60ca28b9-4d26-4eba-8880-5e073491eab3},Bin_5,Segegregate
7,{d461df03-ff8f-4843-a51d-1a62933c9f15},Bin_6,Non - Seggeregated
8,{7f35b8a9-d1db-4b70-9eda-2c02cc03eb9d},Bin_7,Non - seggergated
9,{8dfb4f38-29d0-488f-8bbc-750537655bcf},Bin_8,Non - Segregated


In [12]:

import sqlite3
import pandas as pd

con = sqlite3.connect(str(GPKG_PATH))

# Inspect the attachment records
attachments = pd.read_sql_query(
    "SELECT fid, note_layer, note_uuid, media, description, timestamp "
    "FROM notes_attachments",
    con
)

print("Total attachment records:", len(attachments))
print("\nAttachment table sample:")
display(attachments.head(15))

print("\nAttachment layer values:")
print(attachments["note_layer"].value_counts(dropna=False))

print("\nSample media values:")
print(attachments["media"].head(20).tolist())

# Compare attachment UUIDs with waste-bin observation UUIDs
print("\nObservation UUID sample:")
print(points["uuid"].head(10).tolist())

con.close()


Total attachment records: 37

Attachment table sample:


,fid,note_layer,note_uuid,media,description,timestamp
0,1,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{be965bb7-aeb2-45c6-a5c1-6d91de88a90e},video/VIDEO_20260901164737063.mp4,None,2026-09-01T16:47:37.153Z
1,2,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{ebe25330-4aab-4dd0-b78a-63b47fc9d8fb},video/VIDEO_20260901164955990.mp4,None,2026-09-01T16:49:56.012Z
2,3,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{7234b314-4271-456a-833d-890c5bde1b09},video/VIDEO_20260901165441188.mp4,None,2026-09-01T16:54:41.223Z
3,4,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{92b5c328-1fe9-42d6-8719-d4b63aed3281},video/VIDEO_20260901165812080.mp4,None,2026-09-01T16:58:12.101Z
4,5,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{60ca28b9-4d26-4eba-8880-5e073491eab3},video/VIDEO_20260902172441409.mp4,None,2026-09-02T17:24:41.433Z
5,6,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{d461df03-ff8f-4843-a51d-1a62933c9f15},video/VIDEO_20260902174323816.mp4,None,2026-09-02T17:43:23.903Z
6,7,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{7f35b8a9-d1db-4b70-9eda-2c02cc03eb9d},video/VIDEO_20260902174636653.mp4,None,2026-09-02T17:46:36.691Z
7,8,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{8dfb4f38-29d0-488f-8bbc-750537655bcf},video/VIDEO_20260902174759664.mp4,None,2026-09-02T17:47:59.684Z
8,9,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{ed82c39d-60fa-4ba4-816c-1512ae75059a},video/VIDEO_20260902174934690.mp4,None,2026-09-02T17:49:34.711Z
9,10,Notes___Point_886e4399_19ca_47de_b0f0_36d6b048...,{93664901-179b-4b24-82a6-318f44ea38ee},video/VIDEO_20260902175205554.mp4,None,2026-09-02T17:52:05.575Z



Attachment layer values:
note_layer
Notes___Point_886e4399_19ca_47de_b0f0_36d6b0482466    37
Name: count, dtype: int64

Sample media values:
['video/VIDEO_20260901164737063.mp4', 'video/VIDEO_20260901164955990.mp4', 'video/VIDEO_20260901165441188.mp4', 'video/VIDEO_20260901165812080.mp4', 'video/VIDEO_20260902172441409.mp4', 'video/VIDEO_20260902174323816.mp4', 'video/VIDEO_20260902174636653.mp4', 'video/VIDEO_20260902174759664.mp4', 'video/VIDEO_20260902174934690.mp4', 'video/VIDEO_20260902175205554.mp4', 'video/VIDEO_20260902175448667.mp4', 'video/VIDEO_20260902175745859.mp4', 'video/VIDEO_20260902180557177.mp4', 'video/VIDEO_20260902181112033.mp4', 'video/VIDEO_20260902181145510.mp4', 'video/VIDEO_20260902181318156.mp4', 'video/VIDEO_20260902181547695.mp4', 'video/VIDEO_20260902182037392.mp4', 'video/VIDEO_20260902182928489.mp4', 'video/VIDEO_20260902182947929.mp4']

Observation UUID sample:
['{be965bb7-aeb2-45c6-a5c1-6d91de88a90e}', '{ebe25330-4aab-4dd0-b78a-63b47fc9d8fb}', '{7234

In [15]:

from pathlib import Path
import pandas as pd

# Normalize UUIDs so the observations and attachments match
points["uuid_clean"] = (
    points["uuid"].astype(str).str.strip().str.strip("{}").str.lower()
)

attachments["uuid_clean"] = (
    attachments["note_uuid"].astype(str).str.strip().str.strip("{}").str.lower()
)

# Match each attachment to its observation
matched = points.merge(
    attachments[["uuid_clean", "media", "timestamp"]],
    on="uuid_clean",
    how="left",
    suffixes=("_point", "_attachment")
)

print("Total observations:", len(points))
print("Observations with attachments:", matched["media"].notna().sum())
print("Observations without attachments:", matched["media"].isna().sum())

display(matched[["title", "note", "uuid", "media"]].head(15))


Total observations: 43
Observations with attachments: 37
Observations without attachments: 6


,title,note,uuid,media
0,Bin_1,Segregated,{be965bb7-aeb2-45c6-a5c1-6d91de88a90e},video/VIDEO_20260901164737063.mp4
1,Bin_2,Segregated,{ebe25330-4aab-4dd0-b78a-63b47fc9d8fb},video/VIDEO_20260901164955990.mp4
2,Bin_3,Non - Segregated,{7234b314-4271-456a-833d-890c5bde1b09},video/VIDEO_20260901165441188.mp4
3,Bin_4,Non - Segregated,{92b5c328-1fe9-42d6-8719-d4b63aed3281},video/VIDEO_20260901165812080.mp4
4,NaN,NaN,{04bb926e-1998-4f28-859e-5c303b1fb1b3},NaN
5,NaN,NaN,{16dfbf27-c878-40e1-bb5a-bfc9358b2eb0},NaN
6,Bin_5,Segegregate,{60ca28b9-4d26-4eba-8880-5e073491eab3},video/VIDEO_20260902172441409.mp4
7,Bin_6,Non - Seggeregated,{d461df03-ff8f-4843-a51d-1a62933c9f15},video/VIDEO_20260902174323816.mp4
8,Bin_7,Non - seggergated,{7f35b8a9-d1db-4b70-9eda-2c02cc03eb9d},video/VIDEO_20260902174636653.mp4
9,Bin_8,Non - Segregated,{8dfb4f38-29d0-488f-8bbc-750537655bcf},video/VIDEO_20260902174759664.mp4


In [16]:

from pathlib import Path

# Check likely locations for each relative media path
base_folders = [
    Path(GPKG_PATH).parent,
    Path(PHOTO_DIR)
]

def find_media_file(media):
    if pd.isna(media):
        return None

    relative_path = Path(str(media))

    # Check each base folder
    for base in base_folders:
        candidate = base / relative_path
        if candidate.is_file():
            return str(candidate.resolve())

    # Search for the exact filename under each base folder
    for base in base_folders:
        if base.exists():
            found = list(base.rglob(relative_path.name))
            if found:
                return str(found[0].resolve())

    return None

matched["media_path"] = matched["media"].apply(find_media_file)

print("Matched observations:", len(matched))
print("Video files found:", matched["media_path"].notna().sum())
print("Video files missing:", matched["media"].notna().sum()
      - matched["media_path"].notna().sum())

display(matched.loc[
    matched["media"].notna(),
    ["title", "media", "media_path"]
].head(15))


Matched observations: 43
Video files found: 0
Video files missing: 37


,title,media,media_path
0,Bin_1,video/VIDEO_20260901164737063.mp4,None
1,Bin_2,video/VIDEO_20260901164955990.mp4,None
2,Bin_3,video/VIDEO_20260901165441188.mp4,None
3,Bin_4,video/VIDEO_20260901165812080.mp4,None
6,Bin_5,video/VIDEO_20260902172441409.mp4,None
7,Bin_6,video/VIDEO_20260902174323816.mp4,None
8,Bin_7,video/VIDEO_20260902174636653.mp4,None
9,Bin_8,video/VIDEO_20260902174759664.mp4,None
10,Bin_9,video/VIDEO_20260902174934690.mp4,None
11,Bin_10,video/VIDEO_20260902175205554.mp4,None


In [17]:

import sqlite3
from pathlib import Path

print("GeoPackage:", GPKG_PATH)
print("GeoPackage exists:", Path(GPKG_PATH).exists())
print("Photo folder:", PHOTO_DIR)
print("Photo folder exists:", Path(PHOTO_DIR).exists())

# Show folders and files near the GeoPackage
project_folder = Path(GPKG_PATH).parent

print("\nFiles/folders beside the GeoPackage:")
for item in list(project_folder.iterdir())[:50]:
    print(("DIR  " if item.is_dir() else "FILE "), item.name)

# Check whether the attachment table stores actual video data
con = sqlite3.connect(str(GPKG_PATH))
schema = pd.read_sql_query(
    "PRAGMA table_info(notes_attachments)", con
)
print("\nAttachment table structure:")
display(schema)

# Check whether any other tables may contain media data
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'", con
)
print("\nTables mentioning media, photo, or attachment:")
print([
    name for name in tables["name"]
    if any(word in name.lower()
           for word in ["media", "photo", "attach", "image"])
])

con.close()


GeoPackage: C:\Users\jrana\Downloads\notes.gpkg
GeoPackage exists: True
Photo folder: C:\Users\jrana\Desktop\PHOTOS
Photo folder exists: True

Files/folders beside the GeoPackage:
FILE  02_Tulip Mania.pptx.pdf
FILE  03_Introduction to Macroeconomics.pptx.pdf
FILE  1-s2.0-S0167624524000350-main.pdf
FILE  1-s2.0-S0304393297000421-main.pdf
FILE  1-s2.0-S1703494919300726-main.pdf
FILE  10.4324_9781003348498-18_chapterpdf.pdf
FILE  2011-IndiaStateDist-0000.xlsx
FILE  Anthropologist_CS23B004 (1).docx
DIR   BCom_Vid
FILE  Bengaluru_boundary.geojson
FILE  BIS Bootcamp_Final Poster.png
FILE  Bus_vs_Metro_India_Research_Paper.pdf
FILE  CAER Assignment 3.pdf
FILE  CAER ASSIGNMENT 4 (1).pdf
FILE  CAER ASSIGNMENT 4.pdf
FILE  cdc_88889_DS1.pdf
FILE  Co Creator_VENKATESH_CE23B036.docx
FILE  Copy of Ready to Run Excld_RRBs v1 (1).xlsx
FILE  crisil-inclusix-financial-inclusion-surges-driven-by-Jan-Dhan-yojana.pdf
FILE  datafile.xls
FILE  Data_Scientist_CS23B025.docx
DIR   DC
FILE  desktop.ini
FILE  Dis

,cid,name,type,notnull,dflt_value,pk
0,0,fid,INTEGER,1,None,1
1,1,note_layer,TEXT,0,None,0
2,2,note_uuid,TEXT,0,None,0
3,3,media,TEXT,0,None,0
4,4,description,TEXT,0,None,0
5,5,timestamp,DATETIME,0,None,0



Tables mentioning media, photo, or attachment:
['notes_attachments']


In [18]:

from pathlib import Path

photo_folder = Path(PHOTO_DIR)

all_files = [p for p in photo_folder.rglob("*") if p.is_file()]

print("Total files in PHOTOS folder:", len(all_files))
print("\nFirst 40 files:")
for p in all_files[:40]:
    print(p)

print("\nVideo files:")
videos = [
    p for p in all_files
    if p.suffix.lower() in [".mp4", ".mov", ".avi", ".m4v"]
]
print("Total videos:", len(videos))
for p in videos[:30]:
    print(p)

print("\nImage files:")
images = [
    p for p in all_files
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
]
print("Total images:", len(images))
for p in images[:15]:
    print(p)


Total files in PHOTOS folder: 0

First 40 files:

Video files:
Total videos: 0

Image files:
Total images: 0


In [19]:

from pathlib import Path

# Search Downloads for likely QField project archives and media folders
downloads = Path.home() / "Downloads"

print("Searching:", downloads)

for p in downloads.rglob("*"):
    if p.is_file() and (
        p.suffix.lower() in [".qgz", ".qgs", ".zip", ".mp4", ".jpg", ".jpeg", ".png"]
        or "qfield" in p.name.lower()
        or "video_" in p.name.lower()
    ):
        print(p)


Searching: C:\Users\jrana\Downloads
C:\Users\jrana\Downloads\BIS Bootcamp_Final Poster.png
C:\Users\jrana\Downloads\drive-download-20260606T042758Z-3-001.zip
C:\Users\jrana\Downloads\Gemini_Generated_Image_uf121fuf121fuf12.png
C:\Users\jrana\Downloads\IIM_Sambhalpur.jpeg
C:\Users\jrana\Downloads\image.png
C:\Users\jrana\Downloads\indianpincodes.csv.zip
C:\Users\jrana\Downloads\innerpageb.jpg (1).jpeg
C:\Users\jrana\Downloads\innerpageb.jpg.jpeg
C:\Users\jrana\Downloads\Olivier Blanchard - Macroeconomics (8th Edition)-Pearson (2020).pdf.zip
C:\Users\jrana\Downloads\Waste_Bin_QField_Question1_Complete.ipynb
C:\Users\jrana\Downloads\WhatsApp Image 2026-06-04 at 09.20.12 (1).jpeg
C:\Users\jrana\Downloads\WhatsApp Image 2026-06-04 at 09.20.12.jpeg
C:\Users\jrana\Downloads\WhatsApp Image 2026-06-05 at 11.43.22.jpeg
C:\Users\jrana\Downloads\WhatsApp Image 2026-07-31 at 13.32.24.jpeg
C:\Users\jrana\Downloads\WhatsApp Image 2026-08-27 at 00.58.20.jpeg
C:\Users\jrana\Downloads\WhatsApp Image 202

In [20]:

from pathlib import Path
import zipfile

zip_path = Path.home() / "Downloads" / "drive-download-20260606T042758Z-3-001.zip"

if zip_path.exists() and zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        files = z.namelist()

    print("ZIP file:", zip_path.name)
    print("Total items:", len(files))
    print("\nFiles related to QField, videos, photos, or GeoPackages:")

    relevant = [
        f for f in files
        if any(word in f.lower() for word in [
            "qfield", "video_", ".mp4", ".mov",
            ".jpg", ".jpeg", ".png", ".gpkg", "dcim"
        ])
    ]

    if relevant:
        for f in relevant[:100]:
            print(f)
    else:
        print("No matching media or project filenames found.")
else:
    print("ZIP archive not found or not a valid ZIP file.")


ZIP file: drive-download-20260606T042758Z-3-001.zip
Total items: 7

Files related to QField, videos, photos, or GeoPackages:
No matching media or project filenames found.


## 4. Build the interactive map

Each popup displays the title, notes, collector, coordinates and available photos. User supplied text is HTML escaped. Photographs receive unique filenames and relative `photos/` URLs. The map includes Layer Control, a distance scale and an automatically fitted view.

In [22]:

import folium
import pandas as pd
import html
import hashlib
import shutil
from pathlib import Path
from urllib.parse import quote

# Output folders
OUTPUT_DIR = Path(OUTPUT_DIR)
WEB_VIDEOS = OUTPUT_DIR / "videos"
WEB_VIDEOS.mkdir(parents=True, exist_ok=True)

# Normalize UUIDs
def clean_uuid(value):
    return str(value).strip().strip("{}").lower()

# Build a lookup from observation UUID to an available video file.
# 'matched' is the dataframe created in the previous steps.
video_lookup = {}

for _, row in matched.iterrows():
    media_path = row.get("media_path")

    if pd.notna(media_path) and media_path:
        video_lookup[clean_uuid(row["uuid"])] = Path(media_path)

# Create map
centre = [
    float(points.geometry.y.mean()),
    float(points.geometry.x.mean())
]

m = folium.Map(
    location=centre,
    zoom_start=14,
    tiles="OpenStreetMap",
    control_scale=True
)

my_layer = folium.FeatureGroup(
    name=f"Collected by {MY_NAME}",
    show=True
)
other_layer = folium.FeatureGroup(
    name="Other credited students",
    show=True
)

video_manifest = []

for index, rec in enumerate(records, start=1):
    pt = rec["point"]
    name = rec["collector"]
    title = str(rec.get("title", "Waste Bin"))
    description = str(rec.get("description", "No field note available"))
    record_id = str(rec.get("id", ""))

    popup_parts = [
        '<div style="width:280px;max-height:420px;overflow:auto;'
        'font:14px/1.45 Arial">',
        f'<h3 style="margin:0 0 8px">{html.escape(title)}</h3>',
        f'<p><b>Field note:</b> {html.escape(description)}</p>',
        f'<p><b>Collected by:</b> {html.escape(str(name))}</p>',
        f'<p><b>Record:</b> {html.escape(record_id)}<br>'
        f'<b>Coordinates:</b> {pt.y:.6f}, {pt.x:.6f}</p>'
    ]

    # Find the video attached to this observation.
    source = video_lookup.get(clean_uuid(record_id))

    if source and source.is_file():
        digest = hashlib.sha256(
            str(source).encode("utf-8")
        ).hexdigest()[:12]

        output_name = f"bin_{index:04d}_{digest}{source.suffix.lower()}"
        destination = WEB_VIDEOS / output_name
        shutil.copy2(source, destination)

        relative_url = "videos/" + quote(output_name)

        popup_parts.append(
            '<video controls preload="metadata" '
            'style="width:100%;max-height:240px;margin:8px 0">'
            f'<source src="{relative_url}" type="video/mp4">'
            'Your browser does not support video playback.'
            '</video>'
        )

        video_manifest.append({
            "record": record_id,
            "collector": name,
            "source_filename": source.name,
            "website_video": relative_url
        })
    else:
        popup_parts.append(
            '<p><i>Video unavailable: the referenced file has not '
            'been found on this computer.</i></p>'
        )

    popup_parts.append("</div>")

    layer = (
        my_layer
        if str(name).strip().casefold() == MY_NAME.strip().casefold()
        else other_layer
    )

    folium.Marker(
        [pt.y, pt.x],
        tooltip=title,
        popup=folium.Popup("".join(popup_parts), max_width=320),
        icon=folium.Icon(
            color="green" if layer is my_layer else "blue",
            icon="play",
            prefix="fa"
        )
    ).add_to(layer)

my_layer.add_to(m)

if any(
    str(r["collector"]).strip().casefold()
    != MY_NAME.strip().casefold()
    for r in records
):
    other_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

if len(points) > 1:
    west, south, east, north = points.total_bounds
    m.fit_bounds(
        [[south, west], [north, east]],
        padding=[20, 20],
        max_zoom=17
    )

html_path = OUTPUT_DIR / "waste_bin_map.html"
m.save(str(html_path))

pd.DataFrame(video_manifest).to_csv(
    OUTPUT_DIR / "video_manifest.csv",
    index=False
)

print("Map saved to:", html_path)
print("Videos copied:", len(video_manifest))
print("Video folder:", WEB_VIDEOS)

display(m)


Map saved to: C:\Users\jrana\Downloads\waste_bin_webmap\waste_bin_map.html
Videos copied: 0
Video folder: C:\Users\jrana\Downloads\waste_bin_webmap\videos


## 5. Verify and publish

The final check confirms each referenced photo is present and every collector is named. It cannot verify your eventual host's upload: upload the HTML file **and** the complete `photos` subfolder to the same website directory, then inspect the published popups. Include a link to the published map in your portfolio submission.

In [ ]:
missing_outputs = [v["website_video"] for v in video_manifest
                   if not (OUTPUT_DIR / p["website_video"]).is_file()]
if missing_outputs:
    raise FileNotFoundError(f"Copied photos missing: {missing_outputs}")
page_text = html_path.read_text(encoding="utf-8")
if not all(row["collector"] and html.escape(row["collector"]) in page_text for row in records):
    raise AssertionError("A collector attribution is missing from the generated HTML.")
if not all(p["website_video"] in page_text for p in photo_manifest):
    raise AssertionError("A photograph URL is missing from the generated HTML.")
print("Verified", len(records), "attributed observations and", len(photo_manifest), "video links.")
print("Publish this folder with its directory structure:", OUTPUT_DIR)
print("Map file:", html_path.name)
print("Videos directory:", WEB_VIDEOS.name)
print("Video manifest:", "video_manifest.csv")